# Exercise 02. Join

## Connecting to database

In [1]:
import pandas as pd 
import sqlite3 

conn = sqlite3.connect('../data/checking-logs.sqlite')


## Creating new table - datamart

In [2]:
query = """
SELECT c.uid,
       c.labname,
       MIN(c.timestamp) AS first_commit_ts,
       MIN(p.datetime) AS first_view_ts
FROM checker c
LEFT JOIN pageviews p
ON c.uid = p.uid
WHERE c.status = 'ready'
  AND c.numTrials = 1
  AND c.labname IN ('laba04','laba04s','laba05','laba06','laba06s','project1')
  AND c.uid LIKE 'user_%'
GROUP BY c.uid, c.labname
"""
datamart = pd.read_sql(query, conn, parse_dates=['first_commit_ts', 'first_view_ts'])

## Saving to DB

In [3]:
datamart.to_sql('datamart', conn, if_exists='replace', index=False)

140

## Splitting into two dataframes

In [4]:
test = datamart[datamart['first_view_ts'].notna()].copy()
control = datamart[datamart['first_view_ts'].isna()].copy()

avg_ts = test['first_view_ts'].mean()
control['first_view_ts'] = avg_ts

## Saving them to db 

In [ ]:
test.to_sql('test', conn, if_exists='replace', index=False)
control.to_sql('control', conn, if_exists='replace', index=False)


## Closing connection 

In [6]:
conn.close()